# Estudio 06 · Evaluación en la temporada 2026

Preregistro `SEASON_2026`: regla de decisión aprobada por Rosa el 2026-11-08 (commit `e02ebe1`) y guardas, métricas,
lecturas y validación dejando una temporada fuera escritas antes de entrenar (commit `5a6b27e`).

Variantes (todas con orígenes desde 2005, lluvia CHIRPS v3, temperatura y humedad ERA5-Land): **L + 5a** (sin clima),
**b3**, **d1** y **d2** = d1 + `nino12_anom_media3`. Bloques de siempre + `temporada_2026` (entrenado hasta la temporada
2025) + `posterior_2026` (semanas 35 a 41 de 2026, entrenado hasta la temporada 2026; solo se reporta). Las semanas 42 y
43 de 2026 (provisionales) no entran.

**Regla:** el Brier decide sobre 2010, 2015, 2016, 2022, 2023 y 2026 (>= 5 de 6 a favor y 0 en contra; vetos con >= 4
de 6); se adopta con >= 2 horizontes ganados y sin «pierde» ni «compensa» en el Brier, la F1 ni la razón MAE. Se fijó
después de ver los pasos b y d.

In [1]:
import json
import sys
import unittest

import numpy as np
import pandas as pd

from denguekay.config.paths import GOLD, RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.evaluation import protocol as pa
from denguekay.model import probabilistic_output as sp
from denguekay.model import tracking
from denguekay.model.device import xgboost_settings

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase1b_prime as pb
import phase2_calibration as p2
import phase3_onsets as p3
import phase4_enso as p4
import phase6_other_families as p6
import phase_2026 as p26
import phase_b3_full_climate as b3
import phase_b_rain as pbr
import phase_d_weather_windows as pdw
import preregistration as pr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
METRICS = STUDY / "docs" / "metricas" / "temporada_2026.json"
OUT = p26.OUTPUT_DIR
device = xgboost_settings()
inp = p26.inputs_2026()
origins, sources = p26.add_climate(inp)
_, hp = p3._config()
COLS = p26.variant_columns()
print({k: len(v) for k, v in COLS.items()}, "| provisionales:", inp["semanas_provisionales"])
device

{'L + 5a': 9, 'b3': 13, 'd1': 23, 'd2': 24} | provisionales: [42, 43]


{'device': 'cuda', 'n_jobs': 12}

## 1 · Guardas de datos (antes de entrenar)

In [2]:
gold = pd.read_csv(GOLD / "meteo_socio_epi_piura_2017_2025_h2_gold.csv", dtype={"ubigeo": str},
                   float_precision="round_trip")


def run_tests(names):
    suite = unittest.TestSuite(unittest.defaultTestLoader.loadTestsFromName(n) for n in names)
    result = unittest.TextTestRunner(verbosity=0).run(suite)
    return {"tests": names, "corridos": result.testsRun, "fallas": len(result.failures) + len(result.errors),
            "pasa": result.wasSuccessful()}


sys.path.insert(0, str(STUDY / "tests"))
climate_cols = sorted({c for k in ("b3", "d1", "d2") for c in COLS[k]} - set(COLS["L + 5a"]))
used = pd.concat([inp["matrices"][h] for h in p3.H]).origen_cierre.max()
guards = {
    "G40": p26.guard_series(inp),
    "G17": pb.guard_g17(gold, inp["panel"]),
    "G25": pbr.guard_definition(gold, inp["panel"]),
    "G30": b3.guard_temperature_definition(inp["panel"]),
    "G35a": pdw.guard_daily_matches_silver(sources["semanal_diario"], sources["chirps"]),
    "G35b": pdw.guard_consistency(origins),
    "G42": p26.guard_coverage(origins, climate_cols, used),
    "G43_lluvia_media5": pbr.guard_availability(origins, pr.STEP_B_REFERENCE_DAYS),
    "G43_ventanas": pdw.guard_availability(origins),
    "G43_temperatura": b3.guard_temperature_availability(origins),
    "G43_nino": p4.guard_g21(origins[origins.nino12_anom_ultimo.notna()]),
    "G44": run_tests(["test_phase4_enso.TestAvailability.test_no_leakage_by_perturbation",
                      "test_phase_b_rain.TestDefinition.test_no_leakage_by_perturbation",
                      "test_phase_b3_full_climate.TestTemperature.test_no_leakage_by_perturbation",
                      "test_phase_d_weather_windows.TestWindows.test_no_leakage_by_perturbation"]),
}
failed = [k for k, g in guards.items() if not g["pasa"]]
if failed:
    raise RuntimeError(f"Fallan guardas {failed}: se detiene la evaluación. {guards}")
print("último origen usado:", used)
guards

----------------------------------------------------------------------
Ran 4 tests in 0.237s

OK


último origen usado: 2026-10-03 00:00:00


{'G40': {'filas_2005_2025': 71175,
  'sin_par': 0,
  'distintas': {'casos_Dengue': 0, 'umbral_brote_casos': 0, 'brote': 0},
  'filas_agregadas_2025': 65,
  'semanas_agregadas_2025': [53],
  'casos_agregados_2025': 3,
  'agregadas_fuera_de_la_sala': 0,
  'filas_2026': 2665,
  'distritos_2026': 65,
  'semanas_2026': 41,
  '2026_sin_par_con_la_sala': 0,
  '2026_casos_distintos': 0,
  'pasa': True},
 'G17': {'filas_gold': 30485,
  'con_valor': 30095,
  'mismos_vacios': True,
  'diferencia_maxima': 0.0,
  'pasa': True},
 'G25': {'filas_gold': 30485,
  'con_valor': 30095,
  'mismos_vacios': True,
  'diferencia_maxima': 0.0,
  'pasa': True},
 'G30': {'filas': 30485,
  'con_valor': 30095,
  'mismos_vacios': True,
  'diferencia_maxima': 0.0,
  'pasa': True},
 'G35a': {'semanas_silver': 91000,
  'semanas_solo_en_silver': 0,
  'semanas_solo_en_diario': 0,
  'mismos_vacios': True,
  'diferencia_maxima_mm': 0.0,
  'pasa': True},
 'G35b': {'por_fuente': {'precip_total_mm': {'columna_referencia': 'pr

## 2 · Variantes (5 semillas, todos los bloques) y guardas de reproducción

In [3]:
OUT.mkdir(parents=True, exist_ok=True)
saved5 = json.loads((STUDY / "docs" / "metricas" / "fase5.json").read_text(encoding="utf-8"))
saved = {"L + 5a": p2.read_l_predictions(), "b3": pdw.read_b3_predictions(),
         "d1": p26.read_saved(p3.OUTPUT_DIR.parent / "paso_d" / "predicciones_d1.csv.gz")}
variants = {}
reused = []
for name, cols in COLS.items():
    tag = name.replace(" + ", "_").replace(" ", "")
    path = OUT / f"predicciones_{tag}.csv.gz"
    if path.exists():   # saved by an earlier run of this same notebook (same code and data): evaluated again
        v = p26.evaluate_saved(path, inp["serie"])
        reused.append(name)
    else:
        v = p26.run_variant(inp, origins, cols, hp)
        pd.concat([p.assign(semilla=s) for s, p in v["predicciones"].items()], ignore_index=True).to_csv(path, index=False)
    variants[name] = v
    if name in saved:
        guards[f"G41_{name}"] = p26.guard_old_blocks(v["predicciones"], saved[name])
        print(name, guards[f"G41_{name}"])
        if not guards[f"G41_{name}"]["pasa"]:
            raise RuntimeError(f"G41 falla en {name}: se detiene la evaluación")
guards["G29"] = p6.guard_g16_reproduction(variants["L + 5a"]["evaluaciones"], saved5["veredictos"]["5a"]["metricas"])
if not guards["G29"]["pasa"]:
    raise RuntimeError(f"G29 falla: {guards['G29']}")
print("predicciones reutilizadas de la corrida anterior:", reused)
guards["G29"]

L + 5a {'filas': 915525, 'guardadas_sin_par': 0, 'filas_nuevas_en_bloques_guardados': 975, 'diferencia_maxima': {'casos_predichos': 0.0, 'probabilidad_brote': 0.0}, 'tolerancia': 1e-06, 'pasa': True}


b3 {'filas': 915525, 'guardadas_sin_par': 0, 'filas_nuevas_en_bloques_guardados': 975, 'diferencia_maxima': {'casos_predichos': 0.0, 'probabilidad_brote': 0.0}, 'tolerancia': 1e-06, 'pasa': True}


d1 {'filas': 915525, 'guardadas_sin_par': 0, 'filas_nuevas_en_bloques_guardados': 975, 'diferencia_maxima': {'casos_predichos': 0.0, 'probabilidad_brote': 0.0}, 'tolerancia': 1e-06, 'pasa': True}


predicciones reutilizadas de la corrida anterior: ['L + 5a', 'b3', 'd1', 'd2']


{'diferencia_maxima': 4.9451664985156185e-11, 'filas': 18, 'pasa': True}

## 3 · Veredictos preregistrados (Brier, alerta y conteo sobre 6 temporadas, frente a L + 5a)

In [4]:
outbreaks = p26.outbreak_weeks(inp["serie"])
print("Semanas brote de 2026 por horizonte:", outbreaks)
base = variants["L + 5a"]
results = {k: p26.compare_2026(variants[k], base, outbreaks) for k in ("b3", "d1", "d2")}
for k, r in results.items():
    print(k, "| Brier:", r["brier"].veredicto.tolist(), "| alerta:", r["alerta"].veredicto.tolist(),
          "| conteo:", r["conteo"].veredicto.tolist(), "| adopta:", r["adopta"])
adopted = [k for k, r in results.items() if r["adopta"]]
cross = {}
for i, a in enumerate(adopted):
    for b in adopted[i + 1:]:
        cross[f"{b} frente a {a}"] = p26.compare_2026(variants[b], variants[a], outbreaks)
winner = None
if adopted:
    winner = adopted[0]
    for b in adopted[1:]:
        if cross[f"{b} frente a {winner}"]["adopta"]:
            winner = b
print("Adoptadas:", adopted, "| base resultante:", winner or "L + 5a")

Semanas brote de 2026 por horizonte: {2: 89, 3: 89, 4: 89}


b3 | Brier: ['gana', 'gana', 'empate'] | alerta: ['empate', 'empate', 'empate'] | conteo: ['empate', 'empate', 'empate'] | adopta: True
d1 | Brier: ['empate', 'empate', 'empate'] | alerta: ['empate', 'empate', 'empate'] | conteo: ['empate', 'empate', 'empate'] | adopta: False
d2 | Brier: ['empate', 'empate', 'empate'] | alerta: ['empate', 'empate', 'empate'] | conteo: ['empate', 'empate', 'empate'] | adopta: False
Adoptadas: ['b3'] | base resultante: b3


In [5]:
for k, r in results.items():
    for rule in ("brier", "alerta", "conteo"):
        print(f"--- {k} frente a L + 5a · {rule}")
        display(r[rule])
results["b3"]["antelacion"].round(3)

--- b3 frente a L + 5a · brier


,horizonte,veredicto,vetos,vetos_contra_referencia,a_favor_brier,a_favor_auprc,a_favor_f1_050,a_favor_razon_mae,en_contra_brier,en_contra_auprc,en_contra_f1_050,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion,temporadas
0,2,gana,None,None,5,3,1,3,0,2,1,2,1,1,6
1,3,gana,None,auprc,5,4,2,3,0,2,1,2,1,2,6
2,4,empate,None,auprc,4,4,3,3,0,2,1,2,0,1,6


--- b3 frente a L + 5a · alerta


,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,2,empate,False,False,None,brier,1,3,5,3,1,2,0,2,1,1
1,3,empate,False,False,None,"auprc, brier",2,4,5,3,1,2,0,2,1,2
2,4,empate,False,False,None,"auprc, brier",3,4,4,3,1,2,0,2,0,1


--- b3 frente a L + 5a · conteo


,horizonte,veredicto,vetos,vetos_contra_referencia,a_favor_razon_mae,a_favor_razon_rmse,a_favor_f1_050,a_favor_auprc,a_favor_brier,en_contra_razon_mae,en_contra_razon_rmse,en_contra_f1_050,en_contra_auprc,en_contra_brier,a_favor_antelacion,en_contra_antelacion
0,2,empate,None,brier,3,2,1,3,5,2,0,1,2,0,1,1
1,3,empate,None,"auprc, brier",3,2,2,4,5,2,1,1,2,0,1,2
2,4,empate,None,"auprc, brier",3,2,3,4,4,2,1,1,2,0,0,1


... (9 outputs left out)

,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,2,empate,False,False,None,None,2,2,3,2,1,1,0,2,1,1
1,3,empate,False,False,None,None,2,2,3,2,1,2,0,2,0,2
2,4,empate,False,False,None,brier,3,3,4,3,2,2,0,1,1,1


--- d2 frente a L + 5a · conteo


,horizonte,veredicto,vetos,vetos_contra_referencia,a_favor_razon_mae,a_favor_razon_rmse,a_favor_f1_050,a_favor_auprc,a_favor_brier,en_contra_razon_mae,en_contra_razon_rmse,en_contra_f1_050,en_contra_auprc,en_contra_brier,a_favor_antelacion,en_contra_antelacion
0,2,empate,None,None,2,3,2,2,3,2,0,1,1,0,1,1
1,3,empate,None,None,2,3,2,2,3,2,0,1,2,0,0,2
2,4,empate,None,"razon_rmse, brier",3,4,3,3,4,1,0,2,2,0,1,1


,horizonte,episodios,diferencia,ic95_inferior,ic95_superior,diferencia_temporada_2010,diferencia_temporada_2015,diferencia_temporada_2016,diferencia_temporada_2022,diferencia_temporada_2023,diferencia_temporada_2026,episodios_temporada_2010,episodios_temporada_2015,episodios_temporada_2016,episodios_temporada_2022,episodios_temporada_2023,episodios_temporada_2026
0,2,213,0.000,-0.028,0.033,0.067,0.029,-0.029,-0.053,0.000,0.0,30,34,34,38,57,20
1,3,213,-0.014,-0.047,0.014,0.000,0.029,-0.029,-0.053,-0.053,0.1,30,34,34,38,57,20
2,4,213,-0.014,-0.042,0.014,0.000,0.000,0.000,-0.053,-0.018,0.0,30,34,34,38,57,20


## 4 · Métricas por celda (mediana de 5 semillas)

In [6]:
cells = pd.concat({k: p26.cell_metrics(v, inp["serie"]) for k, v in variants.items()}, names=["variante"]).reset_index(level=0)
order = {b: i for i, b in enumerate([*p26.block_names(p26.CELL_SEASONS), "posterior_2026"])}
cells = cells.sort_values(["bloque", "horizonte", "variante"], key=lambda s: s.map(order) if s.name == "bloque" else s)
show = ["variante", "horizonte", "bloque", "positivos", "brier", "bss_climatologia", "auprc", "auprc_bajo_umbral",
        "positivos_bajo_umbral", "f1_050", "razon_mae", "razon_rmse"]
cells[cells.bloque.isin(["temporada_2026", "posterior_2026"])][show].round(3)

,variante,horizonte,bloque,positivos,brier,bss_climatologia,auprc,auprc_bajo_umbral,positivos_bajo_umbral,f1_050,razon_mae,razon_rmse
8,L + 5a,2,temporada_2026,89.0,0.011,0.630,0.791,0.246,29.0,0.658,1.064,0.990
8,b3,2,temporada_2026,89.0,0.010,0.642,0.801,0.275,29.0,0.671,1.113,1.002
8,d1,2,temporada_2026,89.0,0.010,0.645,0.801,0.265,29.0,0.671,1.082,0.977
8,d2,2,temporada_2026,89.0,0.010,0.644,0.801,0.263,29.0,0.676,1.088,0.980
17,L + 5a,3,temporada_2026,89.0,0.012,0.587,0.743,0.159,26.0,0.557,1.099,0.990
17,b3,3,temporada_2026,89.0,0.011,0.607,0.762,0.171,26.0,0.611,1.118,0.988
17,d1,3,temporada_2026,89.0,0.011,0.612,0.771,0.171,26.0,0.610,1.096,0.979
17,d2,3,temporada_2026,89.0,0.011,0.612,0.771,0.169,26.0,0.610,1.098,0.981
26,L + 5a,4,temporada_2026,89.0,0.016,0.463,0.586,0.115,36.0,0.424,1.072,0.978
26,b3,4,temporada_2026,89.0,0.015,0.484,0.614,0.137,36.0,0.455,1.097,0.960


In [7]:
cells[~cells.bloque.isin(["temporada_2026", "posterior_2026"])][show].round(3)

,variante,horizonte,bloque,positivos,brier,bss_climatologia,auprc,auprc_bajo_umbral,positivos_bajo_umbral,f1_050,razon_mae,razon_rmse
1,L + 5a,2,temporada_2010,234.0,0.030,0.523,0.780,0.263,72.0,0.722,0.907,0.899
1,b3,2,temporada_2010,234.0,0.030,0.528,0.789,0.267,72.0,0.709,0.869,0.883
1,d1,2,temporada_2010,234.0,0.030,0.524,0.779,0.223,72.0,0.714,0.854,0.905
1,d2,2,temporada_2010,234.0,0.030,0.524,0.782,0.233,72.0,0.713,0.849,0.901
10,L + 5a,3,temporada_2010,234.0,0.036,0.432,0.706,0.231,84.0,0.631,0.891,0.890
10,b3,3,temporada_2010,234.0,0.035,0.444,0.724,0.255,84.0,0.634,0.841,0.869
10,d1,3,temporada_2010,234.0,0.036,0.437,0.702,0.207,84.0,0.600,0.813,0.872
10,d2,3,temporada_2010,234.0,0.036,0.437,0.706,0.209,84.0,0.612,0.801,0.865
19,L + 5a,4,temporada_2010,234.0,0.042,0.339,0.600,0.250,106.0,0.476,0.876,0.859
19,b3,4,temporada_2010,234.0,0.041,0.357,0.632,0.291,106.0,0.501,0.819,0.839


In [8]:
# HU0007-4 por celda (2022, 2023 y 2026) y antelación en los bloques de 2026 (semilla 17)
med = {k: pa.seed_median([e["metricas"] for e in v["evaluaciones"].values()]) for k, v in variants.items()}
hu = pd.concat({k: sp.hu0007_4_compliance(m[m.bloque.isin(["temporada_2022", "temporada_2023", "temporada_2026"])])
                for k, m in med.items()}, names=["variante"]).reset_index(level=0)
lead_rows = []
for k, v in variants.items():
    t = v["evaluaciones"][pr.SEEDS[0]]["antelacion"]
    for (h, blk), g in t[t.bloque.isin(["temporada_2026", "posterior_2026"])].groupby(["horizonte", "bloque"]):
        lead_rows.append({"variante": k, "horizonte": h, "bloque": blk, "inicios": len(g),
                          "avisados": float(g.alerta_050.ge(pa.MIN_LEAD_TIME).mean()),
                          "avisados_persistencia": float(g.alerta_persistencia.ge(pa.MIN_LEAD_TIME).mean())})
lead26 = pd.DataFrame(lead_rows)
display(hu[["variante", "horizonte", "bloque", "cumple_clasificacion", "cumple_regresion"]])
lead26.round(3)

,variante,horizonte,bloque,cumple_clasificacion,cumple_regresion
16,L + 5a,2,temporada_2022,False,False
17,L + 5a,2,temporada_2023,True,False
19,L + 5a,2,temporada_2026,False,False
36,L + 5a,3,temporada_2022,False,False
37,L + 5a,3,temporada_2023,False,False
39,L + 5a,3,temporada_2026,False,False
56,L + 5a,4,temporada_2022,False,False
57,L + 5a,4,temporada_2023,False,True
59,L + 5a,4,temporada_2026,False,False
16,b3,2,temporada_2022,False,False


,variante,horizonte,bloque,inicios,avisados,avisados_persistencia
0,L + 5a,2,posterior_2026,17,0.176,0.294
1,L + 5a,2,temporada_2026,20,0.100,0.200
2,L + 5a,3,posterior_2026,17,0.059,0.118
3,L + 5a,3,temporada_2026,20,0.050,0.450
4,L + 5a,4,posterior_2026,17,0.118,0.235
5,L + 5a,4,temporada_2026,20,0.100,0.400
6,b3,2,posterior_2026,17,0.176,0.294
7,b3,2,temporada_2026,20,0.100,0.200
8,b3,3,posterior_2026,17,0.059,0.118
9,b3,3,temporada_2026,20,0.150,0.450


## 5 · Lecturas: Niño por origen, lluvia preliminar y semanas provisionales

In [9]:
pred17 = variants["d2"]["predicciones"][pr.SEEDS[0]]
nino = p26.nino_by_origin(origins, pred17)
prelim = p26.prelim_origins(origins, sources["chirps"], pred17)
datos = json.loads((STUDY / "docs" / "metricas" / "datos_2026.json").read_text(encoding="utf-8"))
provisional = datos["semanas_brote"]["provisionales"]
print("Orígenes fuera del rango del Niño:", nino.groupby("bloque").fuera_de_rango.sum().to_dict(),
      "de", nino.groupby("bloque").size().to_dict())
print("Orígenes con lluvia preliminar (b3 / d):", prelim.groupby("bloque")[["b3_semanas_prelim", "d_semanas_prelim"]].apply(lambda d: (d > 0).sum()).to_dict())
print("Semanas provisionales:", provisional)
display(nino.round(2))
prelim[(prelim.b3_semanas_prelim > 0) | (prelim.d_semanas_prelim > 0)]

Orígenes fuera del rango del Niño: {'posterior_2026': 4, 'temporada_2026': 0} de {'posterior_2026': 9, 'temporada_2026': 55}
Orígenes con lluvia preliminar (b3 / d): {'b3_semanas_prelim': {'posterior_2026': 1, 'temporada_2026': 0}, 'd_semanas_prelim': {'posterior_2026': 1, 'temporada_2026': 0}}
Semanas provisionales: {'semanas': 2, 'desde': '2026-10-18', 'hasta': '2026-10-25', 'casos': 2629, 'semanas_brote': 60, 'distritos_con_brote': 37, 'semanas_brote_por_horizonte': {'h2': 60, 'h3': 60, 'h4': 60}, 'alcanza_minimo_30': True}


,bloque,origen_cierre,nino12_anom_media3,min_entrenamiento,max_entrenamiento,fuera_de_rango
0,temporada_2026,2025-08-02,0.47,-1.74,3.11,False
1,temporada_2026,2025-08-09,0.47,-1.74,3.11,False
2,temporada_2026,2025-08-16,0.41,-1.74,3.11,False
3,temporada_2026,2025-08-23,0.41,-1.74,3.11,False
4,temporada_2026,2025-08-30,0.41,-1.74,3.11,False
5,temporada_2026,2025-09-06,0.41,-1.74,3.11,False
6,temporada_2026,2025-09-13,0.43,-1.74,3.11,False
7,temporada_2026,2025-09-20,0.43,-1.74,3.11,False
8,temporada_2026,2025-09-27,0.43,-1.74,3.11,False
9,temporada_2026,2025-10-04,0.43,-1.74,3.11,False


,bloque,origen_cierre,b3_semanas_prelim,d_semanas_prelim
63,posterior_2026,2026-10-03,1,1


## 6 · Validación dejando una temporada fuera (lectura optimista: usa el futuro; solo compara variantes entre sí)

In [10]:
loso = {}
for name, cols in COLS.items():
    tag = name.replace(" + ", "_").replace(" ", "")
    path = OUT / f"fuera_de_temporada_{tag}.csv.gz"
    if path.exists():
        preds = p26.read_saved(path)
    else:
        preds = {s: p26.loso_predictions(inp, origins, cols, hp, s) for s in pr.SEEDS}
        pd.concat([p.assign(semilla=s) for s, p in preds.items()], ignore_index=True).to_csv(path, index=False)
    loso[name] = p26.loso_metrics(preds, inp["serie"])
loso_all = pd.concat(loso, names=["variante"]).reset_index(level=0)
ref = loso_all[loso_all.variante.eq("L + 5a")].set_index(["horizonte", "bloque"])
metric_cols = ["f1_050", "auprc", "brier", "razon_mae", "razon_rmse", "bss_climatologia", "auprc_bajo_umbral"]
diffs = []
for k in ("b3", "d1", "d2"):
    d = loso_all[loso_all.variante.eq(k)].set_index(["horizonte", "bloque"])[metric_cols] - ref[metric_cols]
    diffs.append(d.assign(variante=k).reset_index())
loso_diff = pd.concat(diffs, ignore_index=True)
better = {k: {m: int(((loso_diff.variante.eq(k)) & (loso_diff[m] * pa.DIRECTION.get(m, 1 if m != "razon_rmse" else -1) > 0)).sum())
              for m in ["brier", "auprc", "f1_050", "razon_mae"]} for k in ("b3", "d1", "d2")}
print("Celdas (de 24) en que la variante con clima mejora a L + 5a:", better)
display(loso_all.sort_values(["bloque", "horizonte", "variante"]).round(3))
loso_diff.round(3)

Celdas (de 24) en que la variante con clima mejora a L + 5a: {'b3': {'brier': 19, 'auprc': 19, 'f1_050': 14, 'razon_mae': 16}, 'd1': {'brier': 22, 'auprc': 20, 'f1_050': 14, 'razon_mae': 17}, 'd2': {'brier': 23, 'auprc': 20, 'f1_050': 13, 'razon_mae': 17}}


,variante,horizonte,bloque,positivos,f1_050,auprc,brier,razon_mae,razon_rmse,bss_climatologia,auprc_bajo_umbral
0,L + 5a,2,fuera_2010,234.0,0.731,0.792,0.030,1.022,1.047,0.484,0.282
0,b3,2,fuera_2010,234.0,0.718,0.795,0.029,0.870,0.842,0.492,0.288
0,d1,2,fuera_2010,234.0,0.724,0.796,0.029,0.876,0.839,0.494,0.284
0,d2,2,fuera_2010,234.0,0.723,0.796,0.029,0.870,0.833,0.495,0.282
8,L + 5a,3,fuera_2010,234.0,0.667,0.726,0.036,1.048,1.080,0.379,0.266
8,b3,3,fuera_2010,234.0,0.671,0.734,0.035,0.829,0.819,0.395,0.273
8,d1,3,fuera_2010,234.0,0.674,0.734,0.035,0.837,0.830,0.397,0.271
8,d2,3,fuera_2010,234.0,0.672,0.734,0.035,0.831,0.823,0.399,0.271
16,L + 5a,4,fuera_2010,234.0,0.605,0.641,0.042,1.090,1.092,0.273,0.329
16,b3,4,fuera_2010,234.0,0.609,0.650,0.041,0.824,0.798,0.294,0.307


,horizonte,bloque,f1_050,auprc,brier,razon_mae,razon_rmse,bss_climatologia,auprc_bajo_umbral,variante
0,2,fuera_2010,-0.013,0.003,-0.000,-0.152,-0.205,0.008,0.005,b3
1,2,fuera_2015,-0.008,0.001,-0.000,0.038,0.073,0.000,0.009,b3
2,2,fuera_2016,-0.008,-0.000,0.000,0.009,-0.021,-0.002,-0.002,b3
3,2,fuera_2017,0.009,0.005,-0.001,-0.062,-0.083,0.007,0.051,b3
4,2,fuera_2022,-0.008,0.002,-0.001,-0.078,-0.086,0.013,-0.012,b3
5,2,fuera_2023,0.009,0.004,-0.004,-0.033,-0.086,0.015,0.012,b3
6,2,fuera_2024,-0.001,0.001,-0.001,-0.226,-0.465,0.004,0.008,b3
7,2,fuera_2026,0.057,0.011,-0.000,0.051,0.013,0.012,0.037,b3
8,3,fuera_2010,0.004,0.008,-0.001,-0.219,-0.261,0.016,0.006,b3
9,3,fuera_2015,-0.001,-0.000,0.000,0.016,0.123,-0.002,-0.003,b3


## 7 · Salidas: métricas y MLflow

In [11]:
to_json = lambda d: {k: diag.to_json(x) for k, x in d.items() if isinstance(x, pd.DataFrame)}
content = {
    "dispositivo": device, "guardas": guards, "semanas_brote_2026": outbreaks, "predicciones_reutilizadas": reused,
    "veredictos": {k: {**to_json(r), "adopta": r["adopta"]} for k, r in results.items()},
    "cruces": {k: {**to_json(r), "adopta": r["adopta"]} for k, r in cross.items()},
    "adoptadas": adopted, "base_resultante": winner or "L + 5a",
    "metricas_por_celda": diag.to_json(cells), "hu0007_4": diag.to_json(hu), "antelacion_2026": diag.to_json(lead26),
    "nino_por_origen": diag.to_json(nino), "origenes_con_lluvia_preliminar": diag.to_json(prelim),
    "semanas_provisionales": {"semanas": inp["semanas_provisionales"], **provisional},
    "dejando_una_temporada_fuera": {"metricas": diag.to_json(loso_all), "diferencias_frente_a_L_5a": diag.to_json(loso_diff),
                                    "celdas_que_mejoran": better},
}
content = json.loads(json.dumps(content, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
verdicts = pd.concat({f"{k} · {rule}": r[rule][["horizonte", "veredicto"]] for k, r in results.items()
                      for rule in ("brier", "alerta", "conteo")}, names=["regla"]).reset_index(level=0)
run_id = tracking.log_study(
    "mejora_modelo_temporada_2026", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "temporada_2026", "preregistro": "e02ebe1 + 5a6b27e", "adoptadas": ", ".join(adopted) or "ninguna",
                       "base_resultante": content["base_resultante"],
                       "veredicto": "; ".join(f"{r.regla} h{r.horizonte}: {r.veredicto}" for r in verdicts.itertuples()),
                       **{f"xgb.{k}": str(x) for k, x in device.items()}},
            "metricas": {}},
    artifacts=[METRICS], tables={"veredicto": verdicts},
    description="Estudio 06: evaluación en la temporada 2026 (L + 5a, b3, d1, d2) con la regla del Brier sobre 6 temporadas.")
run_id

C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: estudio 2d22b09851374954a8cd46ed249ea155 en 'denguekay/mejora_modelo_temporada_2026' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'2d22b09851374954a8cd46ed249ea155'